# PDU Styled Plot Generator

This notebook contains the same code as `analysis/generate_pdu_plot.py`.
Run the code cell to regenerate `analysis/pdu_events_plot.png` with the final visual style.

In [ ]:
import re
from pathlib import Path
from typing import Optional

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


def parse_events(main_log_path: Path) -> pd.DataFrame:
    lines = main_log_path.read_text().splitlines()
    records = []
    last_ts = None

    for line in lines:
        ts_match = re.search(r"ts=(\d{4}-\d{2}-\d{2}T[^\s]+)", line)
        if ts_match:
            try:
                last_ts = pd.to_datetime(ts_match.group(1), utc=True)
            except Exception:
                pass

        event_name = None
        source = None
        event_ts = None

        m_control = re.search(r"\[Control\]\s+event=([A-Z_]+)\s+ts=(\d{4}-\d{2}-\d{2}T[^\s]+)", line)
        if m_control:
            event_name = m_control.group(1)
            source = "control"
            event_ts = pd.to_datetime(m_control.group(2), utc=True)
        else:
            m_overload = re.search(r"\[OverloadEvent\]\s+([A-Z_]+)", line)
            if m_overload:
                event_name = m_overload.group(1)
                source = "overload_event"
                event_ts = last_ts
            elif line.startswith("[DVFS]"):
                event_name = "DVFS_APPLY"
                source = "dvfs"
                event_ts = last_ts
            elif "Jobs ended while reduction active" in line:
                event_name = "JOBS_END_RESET"
                source = "control"
                event_ts = last_ts
            elif line.startswith("Resetting DVFS to max frequency"):
                event_name = "DVFS_RESET"
                source = "control"
                event_ts = last_ts

        if event_name and event_ts is not None:
            records.append(
                {
                    "timestamp": event_ts,
                    "event": event_name,
                    "source": source,
                    "line": line,
                }
            )

    if not records:
        return pd.DataFrame(columns=["timestamp", "event", "source", "line"])

    return pd.DataFrame(records).sort_values("timestamp").reset_index(drop=True)


def infer_capacity_w(main_log_path: Path, events: pd.DataFrame) -> Optional[float]:
    text = main_log_path.read_text()

    m_target = re.search(r"target_capacity_w=([0-9]+(?:\.[0-9]+)?)", text)
    if m_target:
        return float(m_target.group(1))

    m_zone = re.search(r"overload handled zone reached:\s*([0-9.]+)W\.\.([0-9.]+)W", text)
    if m_zone:
        low = float(m_zone.group(1))
        high = float(m_zone.group(2))
        return (low + high) / 2.0

    if not events.empty:
        starts = events[events["event"] == "OVERLOAD_START"]
        for _, row in starts.iterrows():
            line = row["line"]
            m = re.search(
                r"total_watts=([0-9]+(?:\.[0-9]+)?)\s+required_reduction_w=([0-9]+(?:\.[0-9]+)?)",
                line,
            )
            if m:
                total = float(m.group(1))
                reduction = float(m.group(2))
                return total - reduction

    return None


def infer_overload_windows(events: pd.DataFrame):
    windows = []
    if events.empty:
        return windows

    events = events.sort_values("timestamp")
    open_start = None
    end_events = {"OVERLOAD_HANDLED", "OVERLOAD_END"}

    for _, row in events.iterrows():
        event = row["event"]
        ts = row["timestamp"]

        if event == "OVERLOAD_START":
            if open_start is None:
                open_start = ts
        elif event in end_events and open_start is not None:
            windows.append((open_start, ts))
            open_start = None

    if open_start is not None:
        windows.append((open_start, events["timestamp"].max()))

    return windows


def main() -> None:
    try:
        analysis_dir = Path(__file__).resolve().parent
    except NameError:
        cwd = Path.cwd()
        analysis_dir = cwd if (cwd / 'pdu_log.csv').exists() else (cwd / 'analysis')
    pdu_path = analysis_dir / "pdu_log.csv"
    main_log_path = analysis_dir / "main_log.txt"
    plot_path = analysis_dir / "pdu_events_plot.png"
    events_csv = analysis_dir / "events_parsed.csv"

    pdu = pd.read_csv(pdu_path)
    pdu["timestamp"] = pd.to_datetime(pdu["timestamp"], utc=True)
    pdu = pdu.sort_values("timestamp").reset_index(drop=True)

    events = parse_events(main_log_path)
    # Keep only the requested key control events.
    if not events.empty:
        keep_events = {"OVERLOAD_START", "DVFS_APPLY", "OVERLOAD_HANDLED", "OVERLOAD_END"}
        events = events[events["event"].isin(keep_events)].copy().reset_index(drop=True)

    # Keep events only inside PDU time range to avoid axis stretching when logs are from different runs.
    tmin, tmax = pdu["timestamp"].min(), pdu["timestamp"].max()
    if not events.empty:
        in_range = (events["timestamp"] >= tmin) & (events["timestamp"] <= tmax)
        events = events[in_range].copy().reset_index(drop=True)
        if not events.empty:
            events.to_csv(events_csv, index=False)
        else:
            pd.DataFrame(columns=["timestamp", "event", "source", "line"]).to_csv(events_csv, index=False)

    # Visual style matched to analysis/experiment_power_wide.pdf.
    plt.rcParams.update(
        {
            "font.family": "DejaVu Serif",
            "font.weight": "bold",
            "axes.labelweight": "bold",
            "axes.titleweight": "bold",
            "axes.labelsize": 26,
            "axes.titlesize": 36,
            "xtick.labelsize": 20,
            "ytick.labelsize": 20,
            "legend.fontsize": 16,
        }
    )

    pdu["minute"] = (pdu["timestamp"] - tmin).dt.total_seconds() / 60.0
    capacity_w = infer_capacity_w(main_log_path, events)
    overload_windows = infer_overload_windows(events)

    fig, ax = plt.subplots(figsize=(20, 6))
    fig.patch.set_facecolor("#e8e8e8")
    ax.set_facecolor("#e8e8e8")
    ax.plot(
        pdu["minute"],
        pdu["total_watts"],
        color="royalblue",
        linewidth=3.2,
        label="With MPR",
        zorder=2,
    )

    if capacity_w is not None:
        ax.axhline(capacity_w, color="black", linestyle=":", linewidth=4.2, label="Capacity", zorder=1)

    if overload_windows:
        for i, (start_ts, end_ts) in enumerate(overload_windows):
            start_min = (start_ts - tmin).total_seconds() / 60.0
            end_min = (end_ts - tmin).total_seconds() / 60.0
            ax.axvspan(
                start_min,
                end_min,
                color="#ef8f8f",
                alpha=0.42,
                label="Overload" if i == 0 else None,
                zorder=0,
            )

    if not events.empty:
        merged = pd.merge_asof(
            events[["timestamp", "event", "source"]].sort_values("timestamp"),
            pdu[["timestamp", "total_watts"]].sort_values("timestamp"),
            on="timestamp",
            direction="nearest",
        )
        merged["minute"] = (merged["timestamp"] - tmin).dt.total_seconds() / 60.0

        event_styles = {
            "OVERLOAD_START": ("#2ca02c", "^"),
            "DVFS_APPLY": ("#9467bd", "s"),
            "OVERLOAD_HANDLED": ("#e377c2", "D"),
            "OVERLOAD_END": ("#8c564b", "v"),
        }

        event_legend_handles = []
        for ev in merged["event"].unique().tolist():
            g = merged[merged["event"] == ev]
            color, marker = event_styles.get(ev, ("#7f7f7f", "o"))
            for _, row in g.iterrows():
                ax.axvline(row["minute"], color=color, alpha=0.14, linewidth=1.2, zorder=1)
            ax.scatter(
                g["minute"],
                g["total_watts"],
                color=color,
                marker=marker,
                s=80,
                edgecolor="black",
                linewidths=0.6,
                zorder=3,
            )
            event_legend_handles.append(
                plt.Line2D(
                    [],
                    [],
                    color="none",
                    marker=marker,
                    markerfacecolor=color,
                    markeredgecolor="black",
                    markersize=9,
                    label=ev,
                )
            )
    else:
        event_legend_handles = []

    ax.set_xlim(0, pdu["minute"].max())
    ypad = 10.0
    ax.set_ylim(pdu["total_watts"].min() - ypad, pdu["total_watts"].max() + ypad)
    ax.set_title("PDU Power Profile with Event Highlights", pad=10)
    ax.set_xlabel("Time (minute)", labelpad=10)
    ax.set_ylabel("Power (W)", labelpad=10)
    ax.grid(True, linestyle="--", linewidth=1.2, color="#bfbfbf", alpha=0.85)

    base_handles = []
    base_handles.append(plt.Line2D([], [], color="royalblue", linewidth=3.2, label="With MPR"))
    if capacity_w is not None:
        base_handles.append(plt.Line2D([], [], color="black", linestyle=":", linewidth=4.2, label="Capacity"))
    if overload_windows:
        base_handles.append(Patch(facecolor="#ef8f8f", alpha=0.42, edgecolor="none", label="Overload"))

    main_legend = ax.legend(
        handles=base_handles,
        loc="upper center",
        ncol=max(1, len(base_handles)),
        frameon=False,
        handlelength=1.5,
        columnspacing=1.0,
        handletextpad=0.4,
        borderaxespad=0.2,
    )
    ax.add_artist(main_legend)

    if event_legend_handles:
        ax.legend(
            handles=event_legend_handles,
            loc="upper left",
            bbox_to_anchor=(1.01, 1.0),
            frameon=False,
            fontsize=12,
            title="Events",
            title_fontsize=13,
            handlelength=1.0,
            labelspacing=0.35,
            borderaxespad=0.2,
        )

    fig.subplots_adjust(right=0.80, top=0.84, bottom=0.18)
    fig.savefig(plot_path, dpi=160)

    print(f"Wrote plot: {plot_path}")
    print(f"PDU range: {tmin} -> {tmax}")
    print(f"Points: {len(pdu)}")
    print(f"In-range events: {len(events)}")


if __name__ == "__main__":
    main()
